<a href="https://colab.research.google.com/github/temariid/ColabFilesSessia1/blob/main/%D0%97%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B51.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

import plotly
import plotly.graph_objects as go
import statsmodels.api as sm

df = pd.read_csv('/content/sample_data/content/multiregress-092022.csv', decimal='.', sep=',')
df
df = df.drop('i', axis = 1)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df.Me,
        y=df.tc,
        mode='markers',
        name='tc(Me)',
        marker=dict(
            color='rgba(0,0,255,1)',
            size = 7
        ),
        opacity=0.8,
    )
)
fig.update_layout(
    title_text="Температура в зависимости от крутящего момента",
    title_font_size=20,
    xaxis_title="Me,Hm",
    yaxis_title="t, C",
)
fig.show()

y = df.tc.to_numpy()
x= df.Me.to_numpy()
X=sm.add_constant(x)
Xt=X.transpose()
B=np.linalg.inv(Xt.dot(X))
print('B:', B, '\n')
a=np.dot(B.dot(Xt), y)
print('Параметры регрессии: a0:', a[0], 'a1:', a[1], '\n')

y_r=X.dot(a)
print('Рассчетные значения y \n', y_r, '\n')

e=y-y_r
print('Остатки е \n', e)


y=df.tc
X=df.Me
X=sm.add_constant(X)
print('y:\n', y, '\n\n X: \n', X, '\n')

model = sm.OLS(y,X)
reg=model.fit()

print("Параметры регрессии: ", reg.params, "\n")
a0=reg.params.const
a1=reg.params.Me

y_r = reg.fittedvalues
e=reg.resid
print("Расчетные(Прогнозные) значения:\n", y_r)
print("Остатки:\n",e)

print("Отчет по регрессии")
print(reg.summary(), '\n\n')

alpha = 0.05
print('Уровень ошибки', alpha,
      '\nУровень значимости слоя двустороннего обратного распределения Стьюдента:', 1-alpha/2,
      '\nУровень значимости для обратного F-распределения:', 1-alpha)
print('Число степеней свободы регрессии:', reg.df_model)
print('Число степеней свободы остатков:', reg.df_resid)

t_inv=stats.t.ppf(1-alpha/2, reg.df_resid) #квантиль t-распределения
print('Табличное значение критерия Стьюдента', t_inv)

F_inv=stats.f.ppf(1-alpha, reg.df_model, reg.df_resid) #квантиль F-распределения
print('Табличное значение критерия Фишера', F_inv)

fig.add_trace(
    go.Scatter(
        x=x,
        y=y_r,
        mode='lines+markers',
        name='лин.модель',
        marker=dict(
            color='rgba(255,0,0,1)',
            size=7
        ),
        opacity=0.8,

    )
)
fig.show()

x_pr=[1,230]
y_pr=reg.predict(x_pr)
print("Точечный прогноз: y=", y_pr[0], 'при x=', x_pr[1], '\n')

Se=np.sqrt(reg.scale)

alpha = 0.01
t_inv=stats.t.ppf(1-alpha/2, reg.df_resid)
print('Табличное значение критерия Стьюдента', t_inv)
U =Se*t_inv*np.sqrt(1+1/len(y)+(x_pr[1]-x.mean())**2/sum((x-x.mean())**2))
print('Ширина доверительного интервала', U, 'при уровне ошибки', alpha)
print('y=', y_pr[0]-U, '...',y_pr[0]+U)

fig.add_trace(
    go.Scatter(
        x=[x_pr[1]],
        y=[y_pr[0]],
        error_y=dict(
            type ='data',
            array=[U],
            visible=True),
        mode='markers',
        name='прогноз',
        marker=dict(
            color='rgba(0,255,0,1)',
            size=9
        ),
        opacity=0.8,
    ),
)
fig.show()

from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split

nn_model = MLPRegressor (hidden_layer_sizes = (50,10,230), activation='tanh', solver ='lbfgs', random_state=1, max_iter=1000)
regr = nn_model.fit(X,y)
y_pr_nn = regr.predict([[1,230]])

fig.add_trace(
    go.Scatter(
        x=[x_pr[1]],
        y=[y_pr_nn[0]],
        mode='markers',
        name='прогноз по нейросети',
        marker=dict(
            color='rgba(0,255,255,1)',
            size = 9
        ),
        opacity = 0.8,
    ),
)
fig.show()




B: [[ 1.87596516e-01 -9.92737573e-04]
 [-9.92737573e-04  6.93340527e-06]] 

Параметры регрессии: a0: 79.38023966891095 a1: 0.017534834058399326 

Рассчетные значения y 
 [81.83511644 83.06255482 83.79901785 79.87121502 80.60767805 81.83511644
 83.06255482 83.79901785 79.87121502 80.60767805 81.83511644 83.06255482
 83.79901785 79.87121502 80.60767805 81.83511644 83.06255482 83.79901785
 79.87121502 80.60767805 81.83511644 83.06255482] 

Остатки е 
 [-1.13511644 -0.26255482 -0.59901785 -0.57121502 -0.40767805 -1.13511644
  0.13744518 -0.09901785  0.42878498 -0.20767805 -0.33511644 -0.76255482
  0.50098215  0.32878498  0.09232195  0.26488356  0.03744518  1.20098215
  0.82878498  0.19232195  0.96488356  0.53744518]
y:
 0     80.7
1     82.8
2     83.2
3     79.3
4     80.2
5     80.7
6     83.2
7     83.7
8     80.3
9     80.4
10    81.5
11    82.3
12    84.3
13    80.2
14    80.7
15    82.1
16    83.1
17    85.0
18    80.7
19    80.8
20    82.8
21    83.6
Name: tc, dtype: float64 

 X: 


Точечный прогноз: y= 83.41325150234286 при x= 230 

Табличное значение критерия Стьюдента 2.845339709776814
Ширина доверительного интервала 1.9172013336190379 при уровне ошибки 0.01
y= 81.49605016872383 ... 85.33045283596189


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but MLPRegressor was fitted with feature names

